# Mexican Market Data Pipeline

Module: Markets and Data

## Lesson summary

This lab turns the Module 1 market-data framework into a practical data pipeline. Students define a data inventory, route provider access through reusable clients, build a clean price matrix, compute returns, and audit quality before modeling {cite}`fabozzi2019foundations,tsay2010analysis`.

The Mexican market is a useful teaching case because it combines exchange rates, inflation, interest rates, government securities, equity indices, ETFs, FIBRAs, and public market proxies. The purpose is not to build a complete institutional database. The purpose is to create a disciplined educational pipeline that can be reproduced, audited, and extended.

The notebook is designed to work in two modes:

- classroom mode with synthetic data, requiring no credentials or network;
- live data mode, where students can enable Banxico, FRED, or Yahoo Finance calls locally {cite}`banxicoSIE2025,fredAPI2025,yfinance2025`.

## Learning objectives

By the end of this lab, students should be able to:

- document market data sources before modeling;
- design a Mexican market source inventory;
- separate raw extraction, standardization, validation, and analysis-ready outputs;
- configure a Banxico SIE request through the shared data client without exposing credentials;
- understand why adjusted prices matter for equity return calculations;
- align price series to a common business calendar;
- compute simple returns, log returns, annualized volatility, and missingness reports;
- separate raw data, clean data, modeling data, metadata, and quality reports.

## Setup

In [ ]:
import os

import numpy as np
import pandas as pd

from src.market_data import MarketDataClient
from src.market_data_quality import (
    align_to_business_calendar,
    annualized_volatility,
    banxico_series_catalog,
    data_quality_report,
    log_returns,
    source_inventory_template,
)

DATA_MODE = os.getenv("DATA_MODE", "offline")

## Suggested Mexican market dataset

A first version of the Mexican market pipeline can include:

| Variable | Example source | Frequency | Use |
| --- | --- | ---: | --- |
| MXN/USD exchange rate | Banxico | Daily | Currency context |
| Overnight policy rate | Banxico | Daily or event-based | Monetary policy |
| CETES rate | Banxico or Cetesdirecto reference | Daily or weekly | Short-term reference rate |
| Inflation | INEGI | Monthly | Real return and macro context |
| Economic activity | INEGI | Monthly | Growth proxy |
| S&P/BMV IPC | BMV, vendor, or public source | Daily | Equity market proxy |
| FIBRA index or sample FIBRAs | BMV, vendor, or public source | Daily | Real estate vehicle proxy |
| Selected ETFs | BMV, vendor, or public source | Daily | Listed portfolio exposure |

Banxico SIE is appropriate for central bank and financial time series, INEGI is appropriate for national statistical indicators, and BMV data products are appropriate for official market data from the Mexican exchange ecosystem {cite}`banxicoSIE2025,inegiAPI2025,bmvMarketData`.

## Source inventory

A data inventory is part of the analytical documentation, not an optional appendix. It should define what each series is, where it comes from, and how it will be used.

In [ ]:
inventory = source_inventory_template()
inventory.loc[0] = {
    "provider": "Yahoo Finance",
    "instrument_or_variable": "^MXX, AMX.MX, WALMEX.MX",
    "frequency": "daily",
    "start": "2021-01-01",
    "end": "2024-12-31",
    "field": "adjusted close",
    "currency": "MXN",
    "calendar": "trading days",
    "known_limitations": "unofficial endpoint and possible rate limits",
}
inventory.loc[1] = {
    "provider": "Banxico SIE",
    "instrument_or_variable": "SF43718, SF60633, SF60648, SF61745",
    "frequency": "daily or auction frequency",
    "start": "2021-01-01",
    "end": "2024-12-31",
    "field": "published value",
    "currency": "MXN or percent",
    "calendar": "Mexican publication calendar",
    "known_limitations": "requires token and has missing dates",
}
inventory.loc[2] = {
    "provider": "INEGI",
    "instrument_or_variable": "Inflation or economic activity indicator",
    "frequency": "monthly",
    "start": "2021-01-01",
    "end": "2024-12-31",
    "field": "published value or index level",
    "currency": "percent, index, or level",
    "calendar": "Mexican statistical publication calendar",
    "known_limitations": "publication lags, revisions, and unit conventions",
}
inventory.loc[3] = {
    "provider": "BMV or licensed market data source",
    "instrument_or_variable": "S&P/BMV IPC, selected ETFs, FIBRAs",
    "frequency": "daily",
    "start": "2021-01-01",
    "end": "2024-12-31",
    "field": "price, index level, or volume",
    "currency": "MXN",
    "calendar": "Mexican trading calendar",
    "known_limitations": "licensing and official data-product access",
}
inventory

## Pipeline stages

A simple Mexican market pipeline can be organized into six stages:

```text
1. Define series inventory.
2. Extract raw data from each source.
3. Store raw data and metadata.
4. Clean and standardize each series.
5. Align dates and frequencies.
6. Export analysis-ready panels.
```

Each stage should produce files that can be inspected independently.

## Raw extraction and standardization

Raw extraction should preserve the original response whenever possible:

```text
data/raw/banxico/mxn_usd_YYYYMMDD.json
data/raw/banxico/cetes_28d_YYYYMMDD.json
data/raw/inegi/inflation_YYYYMMDD.json
data/raw/fred/us_policy_rate_YYYYMMDD.json
data/raw/yfinance/ipc_proxy_YYYYMMDD.csv
```

Raw files should not be edited manually. If a value appears wrong, the correction belongs in a transformation step and should be documented.

After extraction, each series should be standardized into a common structure:

| Column | Description |
| --- | --- |
| `date` | Observation date |
| `value` | Numeric value |
| `series_id` | Internal series identifier |
| `source` | Provider name |
| `provider_id` | Original provider identifier |
| `frequency` | Daily, monthly, quarterly |
| `unit` | Percent, MXN/USD, index level |
| `retrieval_date` | Date when data were downloaded |
| `quality_flag` | Optional validation flag |

This schema makes it easier to combine Banxico, INEGI, FRED, BMV, and public market sources.

## Date alignment for Mexico

The pipeline should use explicit date rules:

- daily market data use trading dates;
- exchange rates use the source observation date;
- monthly inflation is assigned to the official observation period;
- dashboard views may use month-end alignment;
- historical simulations should use publication dates when avoiding look-ahead bias;
- missing values should not be forward-filled unless the variable is economically persistent and the assumption is documented.

The same source data can support different alignment choices, but each output should state which rule was used.

## Banxico catalog

In [ ]:
banxico_series_catalog()

## Banxico request pattern

Provider-specific API logic belongs in `src`, not inside the notebook. The cell below shows the live-data handoff through `MarketDataClient`; it only runs when `DATA_MODE=live` is set locally and `BANXICO_TOKEN` is configured {cite}`banxicoSIE2025`.

In [ ]:
if DATA_MODE == "live":
    client = MarketDataClient()
    banxico_panel = client.banxico.fetch_series_group(
        ["SF43718", "SF60633", "SF60648", "SF61745"],
        start="2021-01-01",
        end="2024-12-31",
    )
else:
    banxico_panel = pd.DataFrame()

banxico_panel.tail()

## Classroom price matrix

The course build does not depend on external APIs. This synthetic block creates a realistic price matrix for classroom diagnostics.

In [ ]:
rng = np.random.default_rng(42)
dates = pd.bdate_range("2024-01-01", periods=260)
tickers = ["MEX_INDEX", "MX_EQUITY_A", "MX_EQUITY_B"]

daily_shocks = rng.normal(
    loc=[0.00025, 0.00035, 0.00020],
    scale=[0.010, 0.015, 0.012],
    size=(len(dates), len(tickers)),
)
prices = pd.DataFrame(
    100 * np.exp(np.cumsum(daily_shocks, axis=0)),
    index=dates,
    columns=tickers,
)

# Inject a few realistic data-quality issues for the lab.
prices.loc[dates[40], "MX_EQUITY_A"] = np.nan
prices.loc[dates[100], "MX_EQUITY_B"] = np.nan
prices.loc[dates[180], "MX_EQUITY_A"] *= 1.25

prices.head()

## Quality report

In [ ]:
data_quality_report(prices)

## Quality checks for the Mexican panel

The Mexican market panel should include automated checks:

```text
1. No duplicated date-series observations.
2. All values are numeric after parsing.
3. Units are documented.
4. Daily series have expected business-day coverage.
5. Monthly series have one observation per reference month.
6. Extreme returns are flagged.
7. Exchange-rate values are positive.
8. Interest-rate values are within plausible bounds.
9. Inflation series does not mix index levels and percent changes.
10. Equity or ETF prices are checked for splits, dividends, and stale values.
```

Quality checks should produce a report, not only a cleaned file.

## Calendar alignment and returns

In [ ]:
aligned_prices = align_to_business_calendar(prices)
returns = log_returns(aligned_prices)

returns.head()

In [ ]:
summary = pd.DataFrame(
    {
        "mean_daily_return": returns.mean(),
        "annualized_volatility": annualized_volatility(returns),
        "minimum_return": returns.min(),
        "maximum_return": returns.max(),
    }
)
summary

## Output datasets

The pipeline can export several datasets depending on the analysis.

| Output | Purpose |
| --- | --- |
| `mx_macro_long.parquet` | Long-format macro dataset |
| `mx_market_prices_long.parquet` | Long-format prices |
| `mx_returns_wide.parquet` | Wide-format returns |
| `mx_monthly_panel.parquet` | Month-end macro-market panel |
| `quality_report.md` | Data quality summary |
| `data_dictionary.yml` | Metadata and definitions |

This separation prevents one dataset from trying to solve every analytical need.

## Educational workflow

A complete educational workflow can follow this sequence:

```text
1. Download MXN/USD exchange rate from Banxico.
2. Download inflation from INEGI.
3. Download a Mexican equity index proxy.
4. Store raw files locally.
5. Standardize each series to date-value-source format.
6. Validate dates, units, missing values, and outliers.
7. Convert prices into returns.
8. Align daily market data to monthly inflation.
9. Build a monthly panel.
10. Export a dashboard-ready dataset.
```

The final dataset should be simple enough to inspect manually and structured enough to support automated analysis.

## Minimum documentation

The pipeline should include:

```text
README.md
data_dictionary.yml
series_inventory.csv
assumptions_log.md
quality_report.md
source_notes.md
```

The documentation should explain what data were used, why those sources were chosen, how the data were downloaded, which transformations were applied, how missing values were treated, and which limitations remain.

## Source-to-model checklist

| Step | Question |
| --- | --- |
| Provider | Is the source official, commercial, open, or unofficial? |
| Field | Are returns based on adjusted close, close, settlement, bid, ask, or mid? |
| Calendar | Which holidays are represented or missing? |
| Missingness | Are gaps isolated, structural, or provider failures? |
| Outliers | Are flagged observations data errors or real stress events? |
| Transformation | Are models using prices, simple returns, log returns, yields, or spreads? |
| Audit trail | Can another analyst reproduce every cleaning decision? |